# Meta AudioCraft: Game SFX Generator
This notebook provides a GUI to generate sound effects using Meta's `AudioGen` model.

In [ ]:
!pip install av --only-binary=:all:
!pip install --no-deps git+https://github.com/facebookresearch/audiocraft
!pip install xformers julius demucs hydra-colorlog pesq pystoi torchdiffeq torchmetrics docopt encodec flashy omegaconf hydra-core num2words sentencepiece
import ipywidgets as widgets

In [ ]:
import os
# Force refresh of the package path
import site
from importlib import reload
reload(site)

import torch
import torchaudio
from audiocraft.models import AudioGen
from audiocraft.data.audio import audio_write
import ipywidgets as widgets
from IPython.display import display, Audio

# Load the AudioGen model
print("Loading AudioGen model... (approx 1.5GB)")
try:
    # Using medium for high quality game SFX
    model = AudioGen.get_pretrained('facebook/audiogen-medium')
    print("✅ Model loaded successfully!")
except Exception as e:
    print(f"❌ Error loading model: {e}\nTry restarting the session (Runtime -> Restart session) and running the setup cell again.")

In [15]:
# Game SFX Generator UI
style = {'description_width': 'initial'}
prompt_text = widgets.Text(value='car crash with glass breaking', placeholder='e.g., sci-fi laser blast', description='SFX Prompt:', style=style, layout={'width': '80%'})
duration_slider = widgets.IntSlider(value=3, min=1, max=10, step=1, description='Duration (sec):', style=style)
cfg_slider = widgets.FloatSlider(value=3.5, min=1.0, max=7.0, step=0.5, description='Creativity (CFG):', style=style)
generate_btn = widgets.Button(description='Generate Sound Effect', button_style='success', layout={'width': '80%', 'margin': '10px 0px'})
output_area = widgets.Output()

def on_generate_clicked(b):
    with output_area:
        output_area.clear_output()
        if not prompt_text.value.strip():
            print("Please enter a prompt.")
            return

        print(f"🔨 Synthesizing: {prompt_text.value}...")
        # Corrected parameter name: cfg_coef (one 'f')
        model.set_generation_params(duration=duration_slider.value, cfg_coef=cfg_slider.value)

        with torch.no_grad():
            wav = model.generate([prompt_text.value], progress=True)

        audio_data = wav[0].cpu()
        display(Audio(audio_data, rate=model.sample_rate))

        # Auto-save to the Colab file browser
        filename = f"sfx_{prompt_text.value.replace(' ', '_')[:20]}"
        audio_write(filename, audio_data, model.sample_rate, strategy='loudness')
        print(f"✅ Saved as {filename}.wav")

generate_btn.on_click(on_generate_clicked)

display(widgets.VBox([widgets.HTML('<h3>AudioGen SFX Maker</h3>'), prompt_text, duration_slider, cfg_slider, generate_btn, output_area]))